In [18]:
import pandas as pd
from pathlib import Path
from collections import Counter

catalog_dir = Path("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/Earthquake_Catalog/")

In [19]:
print("Magnitude type distribution per patch:")
print("="*75)

all_types = Counter()

for f in sorted(catalog_dir.glob("*.csv")):
    df = pd.read_csv(f)
    mc = {
        "Kanto_Japan": 4.4, "Tohoku_Japan": 4.4,
        "Central_Chile": 4.0, "Central_Turkey": 4.3,
        "Central_Nepal": 4.4, "North_Island_NZ": 4.0,
        "Sichuan_China": 4.3, "Southern_Sumatra_Indonesia": 4.5,
        "Western_Australia": 3.0, "Kuch_India": 4.0,
        "Ordos_China": 4.0, "Southern_Norway": 2.5,
    }.get(f.stem, 4.0)
    
    df_trimmed = df[df['mag'] >= mc].copy()
    counts = df_trimmed['magType'].value_counts()
    all_types.update(counts.to_dict())
    
    print(f"\n{f.stem}")
    print(counts.to_string())

print("\n" + "="*75)
print("GLOBAL magnitude type totals (above Mc):")
for k, v in sorted(all_types.items(), key=lambda x: -x[1]):
    print(f"  {str(k):<12} {v:>6}")

Magnitude type distribution per patch:

Central_Chile
magType
mb     998
mwr    130
mwc     53
ml      49
m       40
mww     35
md      21
mwb      7

Central_Nepal
magType
mb     185
mww      8
mwc      3
mwb      1
mwr      1
mw       1
ml       1

Central_Turkey
magType
mb     312
mwr     92
mww     33
mwc      4
ml       1

Kanto_Japan
magType
mb     1665
mwr     154
mww     120
mwc      96
mwb      24
m         2

Kuch_India
magType
mb     24
mwc     2
mwb     2

North_Island_NZ
magType
mb     396
m      335
ml     212
mwr     32
mww     29
mwc      9
mwb      1

Ordos_China
magType
mb    27

Sichuan_China
magType
mb     643
mwc     25
mww      9
mwb      1
ms       1

Southern_Norway
magType
ml    4

Southern_Sumatra_Indonesia
magType
mb     801
mwc    101
mww     52
mwb     15

Tohoku_Japan
magType
mb     2198
mwc     171
mww     169
mwr     111
mwb      38
ms        1
m         1

Western_Australia
magType
mb    16
ml    15

GLOBAL magnitude type totals (above Mc):
  mb        

In [20]:
MC_FINAL = {
    "Kanto_Japan": 4.4, "Tohoku_Japan": 4.4,
    "Central_Chile": 4.0, "Central_Turkey": 4.3,
    "Central_Nepal": 4.4, "North_Island_NZ": 4.0,
    "Sichuan_China": 4.3, "Southern_Sumatra_Indonesia": 4.5,
    "Western_Australia": 3.0, "Kuch_India": 4.0,
    "Ordos_China": 4.0, "Southern_Norway": 2.5,
}

def convert_to_mw(mag, mag_type):
    """
    Convert magnitude to Mw using Scordilis (2006) global relations.
    Returns (mw_value, conversion_flag)
    flag: 'native' = already Mw, 'converted' = converted, 'uncertain' = unreliable
    """
    mt = str(mag_type).lower().strip()

    # Already moment magnitude
    if mt in ['mw', 'mwr', 'mwc', 'mww', 'mwb', 'mwp']:
        return mag, 'native'

    # mb conversion (Scordilis 2006, valid 3.5-6.2)
    if mt in ['mb', 'mb_lg']:
        if 3.5 <= mag <= 6.2:
            mw = 0.85 * mag + 1.03
            return round(mw, 2), 'converted_mb'
        elif mag > 6.2:
            # mb saturates above 6.2 — use as lower bound, flag
            return round(0.85 * 6.2 + 1.03, 2), 'uncertain_mb_saturated'
        else:
            return round(0.85 * mag + 1.03, 2), 'converted_mb'

    # ms conversion (Scordilis 2006)
    if mt in ['ms', 'ms_20']:
        if 3.0 <= mag <= 6.1:
            mw = 0.67 * mag + 2.07
            return round(mw, 2), 'converted_ms'
        elif 6.2 <= mag <= 8.2:
            mw = 0.99 * mag + 0.08
            return round(mw, 2), 'converted_ms'
        else:
            return mag, 'uncertain_ms'

    # ml — no reliable global conversion
    # Use Deichmann (2006) approximation as fallback
    if mt in ['ml', 'md', 'ml_lg']:
        if mag <= 6.0:
            mw = 0.835 * mag + 0.457  # rough approximation
            return round(mw, 2), 'uncertain_ml'
        else:
            return mag, 'uncertain_ml'

    # Ambiguous 'm' — treat as mb if < 6.0
    if mt == 'm':
        if mag < 6.0:
            mw = 0.85 * mag + 1.03
            return round(mw, 2), 'converted_m_as_mb'
        else:
            return mag, 'uncertain_m'

    # Unknown type
    return mag, 'unknown_type'


In [21]:
processed_dir  = Path("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/processed_catalogs/")
homogenised_dir = Path("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/homogenised_catalogs")

In [22]:
summary = []

for f in sorted(processed_dir.glob("*_full.csv")):
    patch = f.stem.replace("_full", "")
    mc = MC_FINAL.get(patch, 4.0)

    df = pd.read_csv(f)
    df['time'] = pd.to_datetime(df['time'], utc=True, format='mixed')

    # Apply conversion
    results = df.apply(
        lambda row: convert_to_mw(row['mag'], row['magType']), axis=1
    )
    df['mw']       = [r[0] for r in results]
    df['mw_flag']  = [r[1] for r in results]

    # Flag distribution
    flag_counts = df['mw_flag'].value_counts()

    # Re-trim to Mc using converted Mw
    # (some mb events may shift above/below Mc after conversion)
    df_final = df[df['mw'] >= mc].copy().reset_index(drop=True)

    # Save
    df_final.to_csv(homogenised_dir / f"{patch}_full_mw.csv", index=False)

    # Also process declustered version
    decl_f = processed_dir / f"{patch}_declustered.csv"
    if decl_f.exists():
        df_d = pd.read_csv(decl_f)
        df_d['time'] = pd.to_datetime(df_d['time'], utc=True, format='mixed')
        results_d = df_d.apply(
            lambda row: convert_to_mw(row['mag'], row['magType']), axis=1
        )
        df_d['mw']      = [r[0] for r in results_d]
        df_d['mw_flag'] = [r[1] for r in results_d]
        df_d = df_d[df_d['mw'] >= mc].copy().reset_index(drop=True)
        df_d.to_csv(homogenised_dir / f"{patch}_declustered_mw.csv", index=False)

    n_native    = flag_counts.get('native', 0)
    n_converted = sum(v for k, v in flag_counts.items() if 'converted' in k)
    n_uncertain = sum(v for k, v in flag_counts.items() if 'uncertain' in k)

    print(f"{patch:<35} "
          f"native={n_native:>5} "
          f"converted={n_converted:>5} "
          f"uncertain={n_uncertain:>5} "
          f"final_n={len(df_final):>5}")

    summary.append({
        'patch': patch,
        'n_before': len(df),
        'n_native': n_native,
        'n_converted': n_converted,
        'n_uncertain': n_uncertain,
        'n_after_retrim': len(df_final),
        'pct_native': round(100 * n_native / len(df), 1),
        'pct_converted': round(100 * n_converted / len(df), 1),
        'pct_uncertain': round(100 * n_uncertain / len(df), 1),
    })


Central_Chile                       native=  225 converted= 1038 uncertain=   70 final_n= 1276
Central_Nepal                       native=   14 converted=  185 uncertain=    1 final_n=  199
Central_Turkey                      native=  129 converted=  312 uncertain=    1 final_n=  441
Kanto_Japan                         native=  394 converted= 1667 uncertain=    0 final_n= 2061
Kuch_India                          native=    4 converted=   24 uncertain=    0 final_n=   28
North_Island_NZ                     native=   71 converted=  731 uncertain=  212 final_n=  869
Ordos_China                         native=    0 converted=   27 uncertain=    0 final_n=   27
Sichuan_China                       native=   35 converted=  644 uncertain=    0 final_n=  679
Southern_Norway                     native=    0 converted=    0 uncertain=    4 final_n=    4
Southern_Sumatra_Indonesia          native=  168 converted=  801 uncertain=    0 final_n=  969
Tohoku_Japan                        native=  489 c

In [23]:
summary_df = pd.DataFrame(summary)
print("\n" + "="*80)
print("MAGNITUDE HOMOGENISATION SUMMARY")
print("="*80)
print(summary_df.to_string(index=False))


MAGNITUDE HOMOGENISATION SUMMARY
                     patch  n_before  n_native  n_converted  n_uncertain  n_after_retrim  pct_native  pct_converted  pct_uncertain
             Central_Chile      1333       225         1038           70            1276        16.9           77.9            5.3
             Central_Nepal       200        14          185            1             199         7.0           92.5            0.5
            Central_Turkey       442       129          312            1             441        29.2           70.6            0.2
               Kanto_Japan      2061       394         1667            0            2061        19.1           80.9            0.0
                Kuch_India        28         4           24            0              28        14.3           85.7            0.0
           North_Island_NZ      1014        71          731          212             869         7.0           72.1           20.9
               Ordos_China        27         0   

## Insights

### Overview

<p>Magnitude homogenisation was performed as the third and final catalog processing step, following Mc trimming and declustering. The goal is to express all events on a common magnitude scale — moment magnitude Mw — so that temporal features computed across patches are physically comparable. USGS ComCat catalogs mix several magnitude types depending on the reporting network, event size, and available data. Using raw reported magnitudes without homogenisation would introduce systematic scale-dependent biases into every magnitude-based feature: event counts, mean magnitude, b-value estimates, and Omori decay parameters would all reflect magnitude scale differences as much as real seismicity differences, directly undermining the cross-regional transfer objective.</p>

### Magnitude Type Landscape

<p>Across all 12 patches above Mc, the catalog contains approximately 10,000 events distributed across 10 distinct magnitude types. Body wave magnitude mb dominates heavily, accounting for 7,265 events (72.3% of the above-Mc catalog). The Mw variants — mwr (regional moment tensor), mwc (centroid moment tensor), mww (W-phase moment tensor), and mwb (broadband body wave) — collectively account for approximately 1,500 events (15%) and are already expressed in moment magnitude requiring no conversion. Local magnitude ml (282 events, 2.8%), surface wave magnitude ms (2 events), duration magnitude md (21 events), and ambiguous m (378 events, 3.8%) make up the remainder.</p>

<p>The dominance of mb is a known characteristic of global catalogs — it is the fastest magnitude to compute from P-wave arrivals and is therefore the default for most network bulletins. However mb saturates around Mw 6.0–6.2 due to the limited period of the measured waves, meaning that events above this threshold are systematically underestimated if reported in mb. Since your Mc values range from 3.0 to 4.5, the majority of your catalog sits in the Mw 4.0–6.0 range where mb and Mw diverge by up to 0.5 magnitude units — a physically meaningful difference that would corrupt magnitude-based temporal features if uncorrected.</p>

### Conversion Method

<p>Magnitude conversion followed the Scordilis (2006) global empirical relations, the standard reference for catalog homogenisation in seismological studies:</p>

<p>mb → Mw: Mw = 0.85 × mb + 1.03 (valid for mb 3.5–6.2)</p>
<p>ms → Mw: Mw = 0.67 × ms + 2.07 (valid for ms 3.0–6.1); Mw = 0.99 × ms + 0.08 (valid for ms 6.2–8.2)</p>

<p>Events already reported as any Mw variant (mwr, mwc, mww, mwb, mwp) were retained without conversion and flagged as native. Ambiguous m-type magnitudes below 6.0 were treated as mb and converted accordingly. Local magnitude ml events were converted using the Deichmann (2006) approximation (Mw = 0.835 × ml + 0.457) and flagged as uncertain given the absence of a globally reliable ml→Mw relation. Duration magnitude md events received the same uncertain flag. mb events above 6.2 where the scale saturates were converted using the mb 6.2 ceiling value and flagged as uncertain to prevent artificial magnitude inflation. Following conversion, all catalogs were re-trimmed to events ≥ Mc in Mw space — events that were marginally above Mc in the original magnitude scale but fell below Mc after conversion were removed, producing a slightly smaller but more accurate above-Mc sample.</p>

### Findings

<p>Conversion rates are high and physically expected across all patches. Nepal (92.5% converted), Sichuan (94.8%), Ordos (100%), and Kutch (85.7%) show the highest conversion rates, reflecting these networks' reliance on mb as their primary reported scale. Japan patches (Kanto 80.9%, Tohoku 81.6%) and Sumatra (82.7%) are similar. Turkey (70.6%) and Chile (77.9%) have higher native Mw fractions reflecting better moment tensor coverage for these well-studied regions. Uncertain fractions are near-zero for most patches — the main exceptions are New Zealand, Western Australia, and Southern Norway where ml dominates.</p>

<p>Three patches warrant specific notes. New Zealand loses 145 events after re-trimming — these are ml events whose converted Mw values fall below the Mc=4.0 threshold. The Deichmann ml→Mw conversion tends to produce lower values than the original ml scale, and events that were marginally above Mc in ml space fall below it in Mw space. The 869 retained events represent the reliable, consistently scaled subset of the NZ catalog. The 145 dropped events are documented but not recoverable without original moment tensor estimates. Western Australia retains only 27 events after homogenisation with 48.4% uncertain — all ml from the Geoscience Australia network. Given this patch's role as a low-seismicity reference rather than a primary training patch, the thin and partially uncertain catalog is acceptable and documented. Southern Norway's 4 events are all ml and 100% uncertain — consistent with the Nordic network's use of local magnitude as its primary scale — but this patch is already excluded from GNN modelling roles and the uncertainty flag is therefore inconsequential for the analysis.</p>

<p>Chile drops from 1,333 to 1,276 events after re-trimming — 57 events fell below Mc=4.0 after mb→Mw conversion. This is physically correct: mb systematically overestimates Mw for smaller events in the 3.5–4.5 range, meaning some events that appeared above Mc in mb space are genuinely below Mc in Mw space. The smaller but more accurate 1,276-event catalog is preferred.</p>